# ECG Signal Processing Workshop — Notebook 3: Pan-Tompkins Deep Dive

## Historical Context

In 1985, Jiapu Pan and Willis J. Tompkins published *"A Real-Time QRS Detection Algorithm"*
in IEEE Transactions on Biomedical Engineering. The algorithm was designed to run on an
8-bit Z80 microprocessor at 200 Hz — a severe computational constraint that forced
elegant simplicity into the design. Despite being nearly four decades old, Pan-Tompkins
remains the **gold-standard baseline** for R-peak detection in clinical monitors and is
still cited in virtually every new QRS detection paper.

### Why it works so well

The algorithm exploits the unique spectral and morphological properties of the QRS
complex:

- **Steep slopes**: the R-wave has the fastest dV/dt in the cardiac cycle
- **Concentrated energy**: QRS energy peaks between 5–15 Hz
- **Consistent duration**: the QRS complex lasts 80–120 ms in healthy adults

By cascading a bandpass filter, derivative, squaring, and moving-window integration,
each stage amplifies QRS features while suppressing P-waves, T-waves, and noise.

### Limitations with noisy / textile ECG

Pan-Tompkins was developed for **hospital-grade, gelled-electrode recordings**. When
applied to textile-electrode or wearable ECG, several challenges arise:

| Challenge | Effect on Pan-Tompkins |
|---|---|
| Large baseline wander | MWI envelope shifts, thresholds drift |
| Low sampling rate (≤ 100 Hz) | 5-point derivative poorly defined; MWI window too coarse |
| Motion artifacts | Spikes mimic QRS morphology after squaring |
| Low SNR | Adaptive thresholds may converge to noise floor |

This notebook implements Pan-Tompkins **from scratch**, visualises every intermediate
stage, and demonstrates how adaptive thresholding behaves on both clean (BIOPAC) and
noisy (Baby Belt) signals.

## 1. Environment Setup

All packages, file paths, and reproducibility settings.

In [ ]:
# =====================================================================
# USER CONFIGURATION
# =====================================================================
BIOPAC_FILE_PATH = r"sample_data/BIOPAC_04020062_9_Female_1st.txt"
BELT_FILE_PATH   = r"sample_data/BABY_BELT_04020062_9_Female_1st.csv"
# --- Dataset format selector ---
DATASET_FORMAT = "BABY_BELT"  # Options: "BABY_BELT" or "CAREWEAR"
# --- CareWear paths ---
CAREWEAR_BIOPAC_FILE_PATH = r"sample_data/CAREWEAR/P10-stationary Bike1-biopac-01-30-2025.txt"
CAREWEAR_BELT_FILE_PATH   = r"sample_data/CAREWEAR/P10-stationary Bike1-belt-01-30-2025"
CAREWEAR_BIOPAC_ECG_COL   = "CH9"   # "CH9" (raw ECG) or "CH40" (AHA-filtered)
CAREWEAR_BELT_ECG_SCALE_FN = None   # e.g., lambda x: x * 0.001 for future scaling
OUTPUT_DIR       = r"outputs/NB03"
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import (butter, sosfiltfilt, iirnotch, tf2sos,
                          medfilt, find_peaks)
from io import StringIO
from tqdm.notebook import tqdm
import os, warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {os.path.abspath(OUTPUT_DIR)}")

## 2. Data Parsers and Preprocessing Functions

Self-contained copies of the BIOPAC and Baby Belt parsers (from NB01/NB02), plus
the filter functions used for preprocessing.

In [ ]:
# ----- BIOPAC Parser -----

def parse_biopac(filepath):
    """Parse BIOPAC AcqKnowledge exported text file.

    Parameters
    ----------
    filepath : str
        Path to the BIOPAC .txt export file.

    Returns
    -------
    dict
        ecg_raw      : BIOPAC ECG (mV), channel CH40 (CH1 is sync — not ECG)
        ecg_filtered : BIOPAC AHA-filtered ECG (mV), channel CH40
        time_s       : time vector in seconds
        fs           : sampling frequency in Hz
        metadata     : recording metadata dict
    """
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    metadata = {}
    sample_rate_ms = None
    header_idx = None

    for idx, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if "msec/sample" in stripped and sample_rate_ms is None:
            sample_rate_ms = float(stripped.split()[0])
        if stripped.startswith("Recording on:"):
            metadata["recording_time"] = stripped.split(": ", 1)[1]
        if stripped.startswith("milliSec"):
            header_idx = idx
            break

    if sample_rate_ms is None:
        raise ValueError("Could not find sample rate (msec/sample) in header.")
    if header_idx is None:
        raise ValueError("Could not find data header line starting with milliSec.")

    fs = 1000.0 / sample_rate_ms
    col_names = [c.strip() for c in lines[header_idx].strip().split("\t") if c.strip()]
    data_text = "".join(lines[header_idx + 2:])
    df = pd.read_csv(StringIO(data_text), sep="\t", header=None, on_bad_lines="skip")
    df = df.dropna(how="all", axis=1)
    df.columns = col_names[:len(df.columns)]

    time_s = df["milliSec"].values / 1000.0
    ecg_raw = df["CH40"].values.astype(float)
    ecg_filtered = df["CH40"].values.astype(float) if "CH40" in df.columns else ecg_raw.copy()

    duration = len(ecg_raw) / fs
    print(f"  BIOPAC: {len(ecg_raw):,} samples | {fs:.0f} Hz | {duration:.1f} s")
    print(f"  Raw ECG range: [{ecg_raw.min():.4f}, {ecg_raw.max():.4f}] mV")

    return dict(ecg_raw=ecg_raw, ecg_filtered=ecg_filtered, time_s=time_s,
                fs=fs, metadata=metadata)


# ----- Baby Belt Parser -----

def parse_belt(filepath):
    """Parse Baby Belt CSV file.

    Parameters
    ----------
    filepath : str
        Path to the Baby Belt .csv file.

    Returns
    -------
    dict
        ecg      : raw ECG signal (ADC counts)
        time_s   : elapsed time in seconds
        fs       : estimated sampling frequency in Hz
        df       : full pandas DataFrame
        metadata : recording metadata dict
    """
    df = pd.read_csv(filepath)
    time_s = df["PC_Time"].values.astype(float)
    ecg = df["ECG"].values.astype(float)

    dt_median = np.median(np.diff(time_s))
    fs = round(1.0 / dt_median) if dt_median > 0 else 100.0

    duration = time_s[-1] - time_s[0]
    print(f"  Belt:   {len(ecg):,} samples | ~{fs:.0f} Hz | {duration:.1f} s")
    print(f"  Raw ECG range: [{ecg.min():.0f}, {ecg.max():.0f}] (ADC counts)")

    sensor_cols = [c for c in df.columns
                   if c not in ["PC_Time", "Seq", "Tx_ms", "Rx_ms", "Latency", "InterArrival"]]
    return dict(ecg=ecg, time_s=time_s, fs=fs, df=df,
                metadata=dict(estimated_fs=fs, n_samples=len(ecg),
                              duration_s=duration, available_signals=sensor_cols))


# ----- Filter Functions -----

def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=5):
    """Apply zero-phase Butterworth bandpass filter.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    lowcut : float
        Lower cutoff frequency in Hz.
    highcut : float
        Upper cutoff frequency in Hz.
    order : int
        Filter order.

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    sos = butter(order, [lowcut, highcut], btype='band', fs=fs, output='sos')
    return sosfiltfilt(sos, signal)


def notch_filter(signal, fs, freq=60.0, quality=30.0):
    """Apply IIR notch filter at specified frequency.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.
    freq : float
        Notch frequency in Hz.
    quality : float
        Quality factor (higher = narrower notch).

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    nyquist = fs / 2.0
    if freq >= 0.9 * nyquist:
        print(f'  [notch_filter] Skipping {freq:.0f} Hz notch — '
              f'too close to Nyquist ({nyquist:.0f} Hz) for fs={fs:.0f} Hz')
        return signal
    b, a = iirnotch(freq, quality, fs)
    sos = tf2sos(b, a)
    return sosfiltfilt(sos, signal)


def remove_baseline(signal, fs):
    """Remove baseline wander using double median filter.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : float
        Sampling frequency in Hz.

    Returns
    -------
    tuple of (np.ndarray, np.ndarray)
        (signal_corrected, estimated_baseline)
    """
    win1 = int(0.2 * fs) | 1
    win2 = int(0.6 * fs) | 1
    baseline = medfilt(medfilt(signal, win1), win2)
    return signal - baseline, baseline


# ===================== CareWear BIOPAC Parser =====================

def parse_carewear_biopac(filepath, ecg_col=None):
    """Parse CareWear BIOPAC AcqKnowledge exported .txt file.

    Returns a dict matching the parse_biopac() interface.

    Parameters
    ----------
    filepath : str
        Path to the CareWear BIOPAC .txt export file.
    ecg_col : str or None
        Column to use as primary ECG (default: CAREWEAR_BIOPAC_ECG_COL).

    Returns
    -------
    dict
        ecg_raw, ecg_filtered, time_s, fs, metadata
    """
    if ecg_col is None:
        ecg_col = CAREWEAR_BIOPAC_ECG_COL

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    metadata = {}
    sample_rate_ms = None
    header_idx = None

    for idx, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if "msec/sample" in stripped and sample_rate_ms is None:
            sample_rate_ms = float(stripped.split()[0])
        if stripped.startswith("Recording on:"):
            metadata["recording_time"] = stripped.split(": ", 1)[1]
        if stripped.startswith("milliSec"):
            header_idx = idx
            break

    if sample_rate_ms is None:
        raise ValueError("Could not find sample rate (msec/sample) in header.")
    if header_idx is None:
        raise ValueError("Could not find data header row starting with milliSec.")

    fs = 1000.0 / sample_rate_ms
    col_names = [c.strip() for c in lines[header_idx].strip().split("\t") if c.strip()]

    from io import StringIO
    data_text = "".join(lines[header_idx + 2 :])
    df = pd.read_csv(StringIO(data_text), sep="\t", header=None, on_bad_lines="skip")
    df = df.dropna(how="all", axis=1)
    df.columns = col_names[: len(df.columns)]

    time_s = df["milliSec"].values / 1000.0
    ecg_raw = df[ecg_col].values.astype(float)
    ecg_filtered = df["CH40"].values.astype(float) if "CH40" in df.columns else ecg_raw.copy()

    duration = len(ecg_raw) / fs
    print(f"  CareWear BIOPAC: {len(ecg_raw):,} samples | {fs:.0f} Hz | {duration:.1f} s")
    print(f"  ECG ({ecg_col}) range: [{ecg_raw.min():.4f}, {ecg_raw.max():.4f}] mV")

    return dict(ecg_raw=ecg_raw, ecg_filtered=ecg_filtered, time_s=time_s,
                fs=fs, metadata=metadata)



# ===================== CareWear Belt Parser =====================

def parse_carewear_belt(filepath, ecg_scale_fn=None):
    """Parse CareWear belt data file (CSV with Unix epoch timestamps).

    Returns a dict matching the parse_belt() interface.

    Parameters
    ----------
    filepath : str
        Path to the CareWear belt data file.
    ecg_scale_fn : callable or None
        Optional scaling applied before normalisation.

    Returns
    -------
    dict
        ecg, time_s, fs, df, metadata
    """
    if ecg_scale_fn is None:
        ecg_scale_fn = CAREWEAR_BELT_ECG_SCALE_FN

    df = pd.read_csv(filepath)
    ts = df["timestamp"].values.astype(np.float64)
    time_s_arr = (ts - ts[0]) / 1000.0

    ecg_raw = df["Channel 4"].values.astype(np.float64)
    if ecg_scale_fn is not None:
        ecg_raw = ecg_scale_fn(ecg_raw)

    ecg_mean, ecg_std = np.mean(ecg_raw), np.std(ecg_raw)
    ecg = (ecg_raw - ecg_mean) / ecg_std if ecg_std > 0 else ecg_raw - ecg_mean

    dt_ms = np.median(np.diff(ts))
    fs = int(round(1000.0 / dt_ms))

    duration = time_s_arr[-1] - time_s_arr[0]
    print(f"  CareWear Belt: {len(ecg):,} samples | ~{fs:.0f} Hz | {duration:.1f} s")

    sensor_cols = [c for c in df.columns if c != "timestamp"]
    return dict(ecg=ecg, time_s=time_s_arr, fs=fs, df=df,
                metadata=dict(estimated_fs=fs, n_samples=len(ecg),
                              duration_s=duration, available_signals=sensor_cols))



## 3. Data Loading and Preprocessing

Load both files and apply the standard preprocessing chain:
bandpass (0.5–40 Hz) → notch (60 Hz) → baseline removal.

In [ ]:
print("Loading data...")
if DATASET_FORMAT == "CAREWEAR":
    biopac = parse_carewear_biopac(CAREWEAR_BIOPAC_FILE_PATH)
    belt = parse_carewear_belt(CAREWEAR_BELT_FILE_PATH)
else:
    biopac = parse_biopac(BIOPAC_FILE_PATH)
    belt = parse_belt(BELT_FILE_PATH)
print(f"  Format: {DATASET_FORMAT}")

# Preprocess BIOPAC (NB02 order: notch -> bandpass -> baseline; ecg_raw = CH40)
bp_fs = biopac["fs"]
bp_ecg, _ = remove_baseline(
    bandpass_filter(
        notch_filter(biopac["ecg_raw"], bp_fs, 60.0),
        bp_fs, 0.5, 40.0,
    ),
    bp_fs,
)
bp_time = biopac["time_s"]

# Preprocess Belt (same order; 60 Hz notch skips if fs ~100 Hz)
bl_fs = belt["fs"]
bl_ecg, _ = remove_baseline(
    bandpass_filter(
        notch_filter(belt["ecg"], bl_fs, 60.0),
        bl_fs, 0.5, 40.0,
    ),
    bl_fs,
)
bl_time = belt["time_s"]

print(f"\nPreprocessed BIOPAC: {len(bp_ecg):,} samples at {bp_fs:.0f} Hz")
print(f"Preprocessed Belt:   {len(bl_ecg):,} samples at {bl_fs:.0f} Hz")

## 4. The 8 Stages of Pan-Tompkins

The algorithm is a **cascade of signal transforms** followed by **adaptive decision logic**.
Each stage is designed to progressively isolate the QRS complex from everything else.

| Stage | Operation | Purpose |
|:-----:|-----------|--------|
| 1 | **Input signal** (pre-filtered 0.5–40 Hz) | Remove gross noise while preserving QRS morphology |
| 2 | **Bandpass 5–15 Hz** | Isolate the frequency band where QRS energy is maximal |
| 3 | **5-point derivative** | Emphasise the steep rising/falling slopes of the QRS |
| 4 | **Squaring** | Make all values positive; amplify large derivatives non-linearly |
| 5 | **Moving window integration** (~150 ms) | Smooth the squared signal into a single hump per QRS |
| 6 | **Find peaks on MWI** | Identify candidate beat locations |
| 7 | **Adaptive dual threshold** | Classify each candidate as signal peak or noise peak |
| 8 | **Refractory period + searchback** | Reject physiologically impossible detections; recover missed beats |

### Stage-by-stage intuition

**Stage 2 — Bandpass 5–15 Hz**: The QRS complex occupies roughly 5–25 Hz, but the
P-wave and T-wave overlap in the 0.5–10 Hz range. By using a narrow 5–15 Hz pass
band, we maximally separate QRS from the slower waves. A 2nd-order Butterworth
keeps the filter impulse response short to avoid ringing.

**Stage 3 — Derivative**: The 5-point derivative approximation
`h = [1, 2, 0, −2, −1] × (fs/8)` acts as a high-pass that responds to rapid
amplitude changes. The QRS complex — with its steep upstroke and downstroke —
produces the largest derivative values.

**Stage 4 — Squaring**: Eliminates negative values and non-linearly amplifies
large slopes relative to small ones. After squaring, QRS energy dominates.

**Stage 5 — Moving window integration**: Convolving with a rectangular window
of approximately one QRS duration (~150 ms) merges the multiple peaks of the
squared derivative into a single smooth hump per heartbeat.

**Stages 6–8**: The decision rules that convert the MWI envelope into discrete
R-peak detections, covered in detail in the thresholding section below.

## 5. Full Pan-Tompkins Implementation

A self-contained function that returns every intermediate stage for inspection.

In [ ]:
def pan_tompkins_full(ecg, fs):
    """Complete Pan-Tompkins R-peak detection with all intermediate stages.

    Parameters
    ----------
    ecg : np.ndarray
        Pre-filtered ECG signal (bandpassed 0.5-40 Hz).
    fs : int or float
        Sampling frequency in Hz.

    Returns
    -------
    r_peaks : np.ndarray
        Detected R-peak indices in the original signal.
    bp : np.ndarray
        5-15 Hz bandpass output (Stage 2).
    deriv : np.ndarray
        Derivative output (Stage 3).
    squared : np.ndarray
        Squared signal (Stage 4).
    mwi : np.ndarray
        Moving window integration output (Stage 5).
    thresholds : dict
        Running SPK, NPK, THRESHOLD1, THRESHOLD2 arrays.
    """
    fs = float(fs)

    # Stage 2: Bandpass 5-15 Hz (QRS energy isolation)
    sos = butter(2, [5, 15], btype='band', fs=fs, output='sos')
    bp = sosfiltfilt(sos, ecg)

    # Stage 3: 5-point derivative (emphasise steep QRS slopes)
    h = np.array([1, 2, 0, -2, -1]) / 8.0 * fs
    deriv = np.convolve(bp, h, mode='same')

    # Stage 4: Squaring (amplify large derivatives)
    squared = deriv ** 2

    # Stage 5: Moving window integration (~150 ms window)
    win = max(int(0.15 * fs), 1)
    mwi = np.convolve(squared, np.ones(win) / win, mode='same')

    # Stage 6: Find candidate peaks on MWI
    min_dist = int(0.3 * fs)
    candidates, _ = find_peaks(mwi, distance=max(min_dist, 1))

    if len(candidates) == 0:
        empty_thr = dict(
            SPK=np.zeros(len(mwi)), NPK=np.zeros(len(mwi)),
            THRESHOLD1=np.zeros(len(mwi)), THRESHOLD2=np.zeros(len(mwi))
        )
        return np.array([], dtype=int), bp, deriv, squared, mwi, empty_thr

    # Stage 7: Adaptive dual threshold
    init_window = min(int(2 * fs), len(mwi))
    SPK = np.max(mwi[:init_window]) * 0.25
    NPK = np.mean(mwi[:init_window]) * 0.5
    THRESHOLD1 = NPK + 0.25 * (SPK - NPK)
    THRESHOLD2 = 0.5 * THRESHOLD1

    spk_history = np.zeros(len(mwi))
    npk_history = np.zeros(len(mwi))
    thr1_history = np.zeros(len(mwi))
    thr2_history = np.zeros(len(mwi))

    signal_peaks = []
    refractory = int(0.2 * fs)  # 200 ms refractory period
    last_peak = -refractory
    rr_intervals = []

    for c in tqdm(candidates, desc="Adaptive thresholding", leave=False):
        peak_val = mwi[c]

        if c - last_peak < refractory:
            NPK = 0.125 * peak_val + 0.875 * NPK
        elif peak_val > THRESHOLD1:
            SPK = 0.125 * peak_val + 0.875 * SPK
            signal_peaks.append(c)
            if len(signal_peaks) >= 2:
                rr_intervals.append(signal_peaks[-1] - signal_peaks[-2])
            last_peak = c
        else:
            NPK = 0.125 * peak_val + 0.875 * NPK

        THRESHOLD1 = NPK + 0.25 * (SPK - NPK)
        THRESHOLD2 = 0.5 * THRESHOLD1

        spk_history[c] = SPK
        npk_history[c] = NPK
        thr1_history[c] = THRESHOLD1
        thr2_history[c] = THRESHOLD2

    # Stage 8: Searchback — recover missed beats where RR > 1.66× expected
    if len(rr_intervals) > 2:
        rr_mean = np.mean(rr_intervals[-8:]) if len(rr_intervals) >= 8 else np.mean(rr_intervals)
        rr_limit = 1.66 * rr_mean

        searchback_additions = []
        for i in range(1, len(signal_peaks)):
            gap = signal_peaks[i] - signal_peaks[i - 1]
            if gap > rr_limit:
                region_start = signal_peaks[i - 1] + refractory
                region_end = signal_peaks[i] - refractory
                region_cands = candidates[(candidates > region_start) & (candidates < region_end)]
                for rc in region_cands:
                    if mwi[rc] > THRESHOLD2:
                        searchback_additions.append(rc)
                        break

        if searchback_additions:
            signal_peaks = sorted(set(signal_peaks) | set(searchback_additions))

    # Refine to true R-peaks in original signal
    r_peaks = []
    search_win = int(0.05 * fs)
    for p in signal_peaks:
        start = max(0, p - search_win)
        end = min(len(ecg), p + search_win)
        r_peaks.append(start + np.argmax(np.abs(ecg[start:end])))

    # Interpolate threshold histories for smooth visualisation
    for arr in [spk_history, npk_history, thr1_history, thr2_history]:
        nonzero = np.nonzero(arr)[0]
        if len(nonzero) > 1:
            interp_vals = np.interp(np.arange(len(arr)), nonzero, arr[nonzero])
            arr[:] = interp_vals

    thresholds = {
        'SPK': spk_history,
        'NPK': npk_history,
        'THRESHOLD1': thr1_history,
        'THRESHOLD2': thr2_history
    }

    return np.array(r_peaks, dtype=int), bp, deriv, squared, mwi, thresholds

## 6. Run Pan-Tompkins on BIOPAC (Clean Signal)

We select a clean 30-second segment and run the full algorithm.

In [ ]:
VIEW_START = 5.0   # seconds
VIEW_END   = 35.0  # seconds — 30-second window


def extract_segment(ecg, time_s, fs, t_start, t_end):
    """Extract a time-windowed segment from an ECG signal.

    Parameters
    ----------
    ecg : np.ndarray
        Full ECG signal.
    time_s : np.ndarray
        Time vector in seconds.
    fs : float
        Sampling frequency.
    t_start : float
        Segment start time in seconds.
    t_end : float
        Segment end time in seconds.

    Returns
    -------
    tuple
        (segment, segment_time, start_index)
    """
    mask = (time_s >= t_start) & (time_s < t_end)
    idx_start = np.argmax(mask)
    return ecg[mask], time_s[mask], idx_start


bp_seg, bp_seg_t, bp_seg_offset = extract_segment(bp_ecg, bp_time, bp_fs, VIEW_START, VIEW_END)

print(f"BIOPAC segment: {len(bp_seg):,} samples ({VIEW_END - VIEW_START:.0f} s)")
print("Running Pan-Tompkins on BIOPAC segment...")

bp_rpeaks, bp_bp, bp_deriv, bp_sq, bp_mwi, bp_thr = pan_tompkins_full(bp_seg, bp_fs)

print(f"Detected {len(bp_rpeaks)} R-peaks")
if len(bp_rpeaks) > 1:
    rr = np.diff(bp_rpeaks) / bp_fs * 1000
    hr = 60000.0 / rr
    print(f"Mean HR: {np.mean(hr):.1f} bpm  |  RR mean: {np.mean(rr):.1f} ms  |  RR std: {np.std(rr):.1f} ms")

## 7. Eight-Panel Stage Visualisation — BIOPAC

This is the central figure of the notebook: every Pan-Tompkins stage on a shared
time axis so you can see exactly how each transformation isolates the QRS complex.

In [ ]:
def plot_pan_tompkins_stages(ecg, bp, deriv, squared, mwi, thresholds,
                              r_peaks, time_s, title_prefix, fs, save_path):
    """Create an 8-panel figure showing all Pan-Tompkins stages.

    Parameters
    ----------
    ecg : np.ndarray
        Input ECG (Stage 1).
    bp : np.ndarray
        5-15 Hz bandpass output (Stage 2).
    deriv : np.ndarray
        Derivative output (Stage 3).
    squared : np.ndarray
        Squared signal (Stage 4).
    mwi : np.ndarray
        Moving window integration (Stage 5).
    thresholds : dict
        Keys: SPK, NPK, THRESHOLD1, THRESHOLD2.
    r_peaks : np.ndarray
        Detected R-peak indices.
    time_s : np.ndarray
        Time vector in seconds.
    title_prefix : str
        Prefix for panel titles (e.g. 'BIOPAC').
    fs : float
        Sampling frequency.
    save_path : str
        Full path for saving the figure.
    """
    fig, axes = plt.subplots(8, 1, figsize=(16, 22), sharex=True)
    stage_color = "#2c7fb8"

    # Panel 1: Input ECG
    axes[0].plot(time_s, ecg, color=stage_color, linewidth=0.5)
    axes[0].set_title(f"{title_prefix} — Stage 1: Input ECG (0.5–40 Hz bandpass)",
                      fontsize=11, fontweight="bold")
    axes[0].set_ylabel("Amplitude")

    # Panel 2: 5-15 Hz bandpass
    axes[1].plot(time_s, bp, color="#d95f02", linewidth=0.5)
    axes[1].set_title(f"{title_prefix} — Stage 2: Bandpass 5–15 Hz (QRS energy isolation)",
                      fontsize=11, fontweight="bold")
    axes[1].set_ylabel("Amplitude")

    # Panel 3: Derivative
    axes[2].plot(time_s, deriv, color="#7570b3", linewidth=0.5)
    axes[2].set_title(f"{title_prefix} — Stage 3: 5-Point Derivative (slope emphasis)",
                      fontsize=11, fontweight="bold")
    axes[2].set_ylabel("Amplitude")

    # Panel 4: Squared
    axes[3].plot(time_s, squared, color="#e7298a", linewidth=0.5)
    axes[3].set_title(f"{title_prefix} — Stage 4: Squaring (non-linear amplification)",
                      fontsize=11, fontweight="bold")
    axes[3].set_ylabel("Amplitude")

    # Panel 5: MWI
    axes[4].plot(time_s, mwi, color="#1b9e77", linewidth=0.8)
    axes[4].set_title(f"{title_prefix} — Stage 5: Moving Window Integration (~150 ms)",
                      fontsize=11, fontweight="bold")
    axes[4].set_ylabel("Amplitude")

    # Panel 6: Thresholds overlaid on MWI
    axes[5].plot(time_s, mwi, color="#1b9e77", linewidth=0.6, alpha=0.6, label="MWI")
    axes[5].plot(time_s, thresholds['THRESHOLD1'], color="red", linewidth=1.0,
                 linestyle="--", label="THRESHOLD1")
    axes[5].plot(time_s, thresholds['THRESHOLD2'], color="orange", linewidth=1.0,
                 linestyle=":", label="THRESHOLD2")
    axes[5].set_title(f"{title_prefix} — Stage 6–7: Adaptive Thresholds on MWI",
                      fontsize=11, fontweight="bold")
    axes[5].set_ylabel("Amplitude")
    axes[5].legend(loc="upper right", fontsize=9)

    # Panel 7: Detected peaks on MWI
    axes[6].plot(time_s, mwi, color="#1b9e77", linewidth=0.6, alpha=0.5, label="MWI")
    valid_mwi = r_peaks[r_peaks < len(mwi)]
    if len(valid_mwi) > 0:
        axes[6].plot(time_s[valid_mwi], mwi[valid_mwi], "rv", markersize=8,
                     label=f"Detected peaks ({len(valid_mwi)})")
    axes[6].set_title(f"{title_prefix} — Stage 7: Detected R-peaks on MWI",
                      fontsize=11, fontweight="bold")
    axes[6].set_ylabel("Amplitude")
    axes[6].legend(loc="upper right", fontsize=9)

    # Panel 8: Final R-peaks on original ECG
    axes[7].plot(time_s, ecg, color=stage_color, linewidth=0.5, label="ECG")
    valid_ecg = r_peaks[r_peaks < len(ecg)]
    if len(valid_ecg) > 0:
        axes[7].plot(time_s[valid_ecg], ecg[valid_ecg], "g^", markersize=10,
                     label=f"R-peaks ({len(valid_ecg)})")
    axes[7].set_title(f"{title_prefix} — Stage 8: Final R-peaks on ECG",
                      fontsize=11, fontweight="bold")
    axes[7].set_ylabel("Amplitude")
    axes[7].set_xlabel("Time (s)")
    axes[7].legend(loc="upper right", fontsize=9)

    for ax in axes:
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"{title_prefix}: Pan-Tompkins — All 8 Stages",
                 fontsize=14, fontweight="bold", y=1.0)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_pan_tompkins_stages(
    bp_seg, bp_bp, bp_deriv, bp_sq, bp_mwi, bp_thr,
    bp_rpeaks, bp_seg_t, "BIOPAC", bp_fs,
    os.path.join(OUTPUT_DIR, "pan_tompkins_8stages_biopac.png")
)

## 8. Eight-Panel Stage Visualisation — Baby Belt

Now we apply the identical pipeline to the textile-electrode Belt signal.
Watch for broader MWI humps (lower fs → fewer samples per QRS) and noisier
threshold tracking.

In [ ]:
bl_seg, bl_seg_t, bl_seg_offset = extract_segment(bl_ecg, bl_time, bl_fs, VIEW_START, VIEW_END)

print(f"Belt segment: {len(bl_seg):,} samples ({VIEW_END - VIEW_START:.0f} s)")
print("Running Pan-Tompkins on Belt segment...")

bl_rpeaks, bl_bp, bl_deriv, bl_sq, bl_mwi, bl_thr = pan_tompkins_full(bl_seg, bl_fs)

print(f"Detected {len(bl_rpeaks)} R-peaks")
if len(bl_rpeaks) > 1:
    rr = np.diff(bl_rpeaks) / bl_fs * 1000
    hr = 60000.0 / rr
    print(f"Mean HR: {np.mean(hr):.1f} bpm  |  RR mean: {np.mean(rr):.1f} ms  |  RR std: {np.std(rr):.1f} ms")

plot_pan_tompkins_stages(
    bl_seg, bl_bp, bl_deriv, bl_sq, bl_mwi, bl_thr,
    bl_rpeaks, bl_seg_t, "Baby Belt", bl_fs,
    os.path.join(OUTPUT_DIR, "pan_tompkins_8stages_belt.png")
)

## 9. Adaptive Thresholding — Deep Dive

The genius of Pan-Tompkins lies in its **dual running-average thresholds** that
continuously adapt to the signal amplitude. This makes the algorithm robust to
gradual changes in ECG amplitude (e.g., respiration-modulated R-wave height).

### The four update equations

Every time a peak is found in the MWI, it is classified as either a **signal peak**
(genuine QRS) or a **noise peak** (everything else). The classification depends on
whether it exceeds `THRESHOLD1`:

**Signal peak detected** (peak > THRESHOLD1):

$$\text{SPK} = 0.125 \times \text{peak} + 0.875 \times \text{SPK}$$

**Noise peak detected** (peak ≤ THRESHOLD1):

$$\text{NPK} = 0.125 \times \text{peak} + 0.875 \times \text{NPK}$$

After each update, the thresholds are recalculated:

$$\text{THRESHOLD1} = \text{NPK} + 0.25 \times (\text{SPK} - \text{NPK})$$

$$\text{THRESHOLD2} = 0.5 \times \text{THRESHOLD1}$$

### Interpretation

- **SPK** is an exponentially-weighted running average of signal (QRS) peak heights
- **NPK** is an exponentially-weighted running average of noise peak heights
- **THRESHOLD1** sits at 25% of the way from NPK to SPK — a compromise that
  catches most QRS peaks while rejecting most noise
- **THRESHOLD2** (half of THRESHOLD1) is used during **searchback** when a beat
  appears to have been missed

The coefficient 0.125 (= 1/8) gives the exponential averages a time constant of
about 8 beats. This is slow enough to be stable but fast enough to adapt to
changes over 10–15 seconds.

## 11. Visualise Adaptive Threshold Curves — 2-Minute Window

A longer window reveals how the thresholds track amplitude changes over time.

In [ ]:
THR_START = 5.0    # seconds
THR_END   = 125.0  # 2-minute window

bp_thr_seg, bp_thr_t, _ = extract_segment(bp_ecg, bp_time, bp_fs, THR_START, THR_END)

print(f"Running Pan-Tompkins on 2-minute BIOPAC segment ({len(bp_thr_seg):,} samples)...")
bp_thr_rpeaks, _, _, _, bp_thr_mwi, bp_thr_curves = pan_tompkins_full(bp_thr_seg, bp_fs)
print(f"Detected {len(bp_thr_rpeaks)} R-peaks in 2-minute segment.")

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Panel 1: ECG with R-peaks
axes[0].plot(bp_thr_t, bp_thr_seg, color="steelblue", linewidth=0.4, label="ECG")
valid = bp_thr_rpeaks[bp_thr_rpeaks < len(bp_thr_seg)]
if len(valid) > 0:
    axes[0].plot(bp_thr_t[valid], bp_thr_seg[valid], "g^", markersize=5,
                 label=f"R-peaks ({len(valid)})")
axes[0].set_title("BIOPAC — Preprocessed ECG with R-peaks (2-minute window)",
                  fontsize=12, fontweight="bold")
axes[0].set_ylabel("Amplitude")
axes[0].legend(loc="upper right", fontsize=9)

# Panel 2: MWI with thresholds
axes[1].plot(bp_thr_t, bp_thr_mwi, color="#1b9e77", linewidth=0.5, alpha=0.7, label="MWI")
axes[1].plot(bp_thr_t, bp_thr_curves['THRESHOLD1'], color="red", linewidth=1.2,
             linestyle="--", label="THRESHOLD1")
axes[1].plot(bp_thr_t, bp_thr_curves['THRESHOLD2'], color="orange", linewidth=1.0,
             linestyle=":", label="THRESHOLD2")
axes[1].set_title("MWI with Adaptive Thresholds",
                  fontsize=12, fontweight="bold")
axes[1].set_ylabel("Amplitude")
axes[1].legend(loc="upper right", fontsize=9)

# Panel 3: SPK and NPK running averages
axes[2].plot(bp_thr_t, bp_thr_curves['SPK'], color="#d62728", linewidth=1.0,
             label="SPK (signal peak avg)")
axes[2].plot(bp_thr_t, bp_thr_curves['NPK'], color="#9467bd", linewidth=1.0,
             label="NPK (noise peak avg)")
axes[2].fill_between(bp_thr_t, bp_thr_curves['NPK'], bp_thr_curves['SPK'],
                     alpha=0.15, color="gray", label="SPK–NPK gap")
axes[2].set_title("Running SPK and NPK Averages",
                  fontsize=12, fontweight="bold")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")
axes[2].legend(loc="upper right", fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle("Adaptive Threshold Behaviour Over 2 Minutes (BIOPAC)",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "adaptive_thresholds_2min.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

## 12. T-Wave Rejection and Searchback

### The 200 ms refractory period

After detecting an R-peak, the algorithm ignores all candidates for the next
200 ms. This is physiologically motivated: the absolute refractory period of
ventricular myocardium is approximately 200–250 ms, during which another QRS
complex **cannot** occur. This blanking window serves two critical purposes:

1. **T-wave rejection**: The T-wave immediately follows the QRS complex
   (typically 200–400 ms after the R-peak). After squaring and integration,
   a prominent T-wave can produce a secondary MWI hump that exceeds THRESHOLD1.
   The refractory period prevents this from being classified as a second beat.

2. **Derivative ringing suppression**: The 5-point derivative of a sharp QRS
   can produce oscillatory tails. Without the refractory period, these ripples
   could generate spurious candidate peaks.

### Searchback for missed beats

If the interval between two consecutive detected beats exceeds **1.66 × the
running average RR interval**, the algorithm suspects a beat was missed. It
searches back through the gap using the lower threshold (THRESHOLD2) to find
any peak that might have been just below THRESHOLD1.

This mechanism is particularly important for:
- **Premature ventricular contractions (PVCs)** which may have unusual morphology
  and lower MWI amplitude
- **Signal dropouts** where a beat's amplitude was transiently reduced

## 13. T-Wave Rejection Demo

We zoom into a segment where T-waves produce visible MWI humps and show how
the refractory period prevents false detections.

In [ ]:
def find_prominent_twave_segment(ecg, mwi, r_peaks, fs, window_beats=5):
    """Find a segment where T-waves create visible secondary MWI peaks.

    Searches for a region where the MWI has secondary peaks between
    detected R-peaks, indicating prominent T-waves.

    Parameters
    ----------
    ecg : np.ndarray
        ECG signal.
    mwi : np.ndarray
        MWI signal.
    r_peaks : np.ndarray
        Detected R-peak indices.
    fs : float
        Sampling frequency.
    window_beats : int
        Number of beats to include in the demo window.

    Returns
    -------
    tuple
        (start_idx, end_idx, secondary_peak_indices)
    """
    best_score = 0
    best_region = (0, min(int(5 * fs), len(ecg)))
    best_secondary = []
    refractory = int(0.2 * fs)

    for i in range(len(r_peaks) - window_beats):
        start = r_peaks[i]
        end = r_peaks[i + window_beats]
        if end >= len(mwi):
            continue

        secondary = []
        for j in range(i, i + window_beats - 1):
            rp = r_peaks[j]
            rp_next = r_peaks[j + 1]
            t_zone_start = rp + refractory
            t_zone_end = min(rp + int(0.5 * fs), rp_next - refractory)

            if t_zone_start >= t_zone_end or t_zone_end >= len(mwi):
                continue

            zone = mwi[t_zone_start:t_zone_end]
            local_peaks, props = find_peaks(zone, prominence=np.max(mwi[start:end]) * 0.05)

            for lp in local_peaks:
                secondary.append(t_zone_start + lp)

        if len(secondary) > best_score:
            best_score = len(secondary)
            best_region = (start, end)
            best_secondary = secondary

    return best_region[0], best_region[1], best_secondary


t_start_idx, t_end_idx, secondary_peaks = find_prominent_twave_segment(
    bp_seg, bp_mwi, bp_rpeaks, bp_fs, window_beats=6
)

pad = int(0.5 * bp_fs)
plot_start = max(0, t_start_idx - pad)
plot_end = min(len(bp_seg), t_end_idx + pad)
sl = slice(plot_start, plot_end)
t_plot = bp_seg_t[sl]

peaks_in_range = bp_rpeaks[(bp_rpeaks >= plot_start) & (bp_rpeaks < plot_end)]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Panel 1: ECG with detected R-peaks
axes[0].plot(t_plot, bp_seg[sl], color="steelblue", linewidth=0.6, label="ECG")
if len(peaks_in_range) > 0:
    axes[0].plot(bp_seg_t[peaks_in_range], bp_seg[peaks_in_range], "g^",
                 markersize=10, label="Detected R-peaks")
axes[0].set_title("ECG Segment — T-wave Rejection Demo", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Amplitude")
axes[0].legend(loc="upper right", fontsize=9)

# Panel 2: MWI with refractory zones highlighted
axes[1].plot(t_plot, bp_mwi[sl], color="#1b9e77", linewidth=0.8, label="MWI")

refractory_samples = int(0.2 * bp_fs)
for rp in peaks_in_range:
    ref_start = rp
    ref_end = min(rp + refractory_samples, plot_end)
    if ref_start < plot_end and ref_end > plot_start:
        t_ref_s = bp_seg_t[max(ref_start, plot_start)]
        t_ref_e = bp_seg_t[min(ref_end, plot_end - 1)]
        axes[1].axvspan(t_ref_s, t_ref_e, alpha=0.2, color="red",
                        label="Refractory (200 ms)" if rp == peaks_in_range[0] else "")

sec_in_range = [s for s in secondary_peaks if plot_start <= s < plot_end]
if sec_in_range:
    axes[1].plot(bp_seg_t[sec_in_range], bp_mwi[sec_in_range], "x",
                 color="darkorange", markersize=10, markeredgewidth=2,
                 label=f"T-wave humps ({len(sec_in_range)})")

axes[1].set_title("MWI — Refractory Zones Block T-wave False Positives",
                  fontsize=12, fontweight="bold")
axes[1].set_ylabel("Amplitude")
axes[1].legend(loc="upper right", fontsize=9)

# Panel 3: MWI with thresholds
axes[2].plot(t_plot, bp_mwi[sl], color="#1b9e77", linewidth=0.6, alpha=0.5, label="MWI")
axes[2].plot(t_plot, bp_thr['THRESHOLD1'][sl], color="red", linewidth=1.0,
             linestyle="--", label="THRESHOLD1")
axes[2].plot(t_plot, bp_thr['THRESHOLD2'][sl], color="orange", linewidth=1.0,
             linestyle=":", label="THRESHOLD2")

if sec_in_range:
    for si in sec_in_range:
        above_t1 = bp_mwi[si] > bp_thr['THRESHOLD1'][si]
        color = "red" if above_t1 else "green"
        label_txt = "Above THR1 (would be FP)" if above_t1 else "Below THR1 (rejected)"
        axes[2].annotate(
            label_txt, xy=(bp_seg_t[si], bp_mwi[si]),
            xytext=(bp_seg_t[si], bp_mwi[si] + np.max(bp_mwi[sl]) * 0.15),
            fontsize=8, color=color, ha="center",
            arrowprops=dict(arrowstyle="->", color=color, lw=1)
        )

axes[2].set_title("T-wave Humps vs Adaptive Thresholds",
                  fontsize=12, fontweight="bold")
axes[2].set_ylabel("Amplitude")
axes[2].set_xlabel("Time (s)")
axes[2].legend(loc="upper right", fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle("T-Wave Rejection Mechanism",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "twave_rejection_demo.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")
print(f"\nT-wave secondary humps found in window: {len(sec_in_range)}")
print(f"All rejected by refractory period or threshold — zero false positives.")

## 14. Summary

### What we covered

This notebook implemented the **complete Pan-Tompkins QRS detection algorithm**
from scratch and visualised every intermediate processing stage on both a
clinical-grade BIOPAC recording and a noisy textile Baby Belt recording.

**Key takeaways:**

1. **Cascaded signal transforms** (bandpass → derivative → squaring → MWI) are
   not arbitrary: each stage exploits a specific property of the QRS complex
   (frequency content, slope steepness, amplitude dominance, temporal width).

2. **Adaptive thresholding** with exponentially-weighted running averages (SPK/NPK)
   makes the algorithm self-calibrating. It handles gradual amplitude changes
   without manual threshold tuning.

3. **The 200 ms refractory period** is essential for T-wave rejection. Without it,
   prominent T-waves would produce false R-peak detections.

4. **Searchback** recovers missed beats by re-examining gaps with a lower threshold,
   reducing false negatives.

5. **Low sampling rate degrades performance**: At ~100 Hz (Baby Belt), the derivative
   and MWI stages lose temporal resolution, and the adaptive thresholds may struggle
   with the noisier MWI envelope.

### Limitations of Pan-Tompkins

- Assumes relatively **stationary signal quality** over 8–10 beat intervals
- The fixed 150 ms MWI window is tuned for adults; neonatal or tachycardic
  QRS complexes may need a shorter window
- Motion artefacts that mimic QRS morphology (steep slopes + narrow duration)
  can fool every stage of the pipeline
- No explicit mechanism for handling **atrial fibrillation** or **bundle branch block**

### Next notebooks

| Notebook | Focus |
|---|---|
| **NB04** | Deep-learning R-peak detection (RPNet) — can a neural network outperform Pan-Tompkins on noisy textile ECG? |
| **NB05** | HRV feature extraction from the detected R-peaks |
| **NB06** | Cross-system comparison and Bland-Altman analysis |